# Phase 3 — Old vs New Annotation Benchmark

Compares model performance when trained on **original human annotations** (`text_human`) vs
**updated LLM annotations** (`text_llm`) from Phase 2.

All configurations use **DistilBERT (text) + DenseNet121 (image)** with cross-modal attention fusion.
Hyperparameters vary across configs. Each config is trained twice — once per annotation set —
on **identical data splits** for a fair comparison. **15 epochs per run (patience=5), CPU only.**




In [1]:
# ── Cell 1: Setup, imports, and configuration ──────────────────────────────
import os
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import DenseNet121_Weights
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Force CPU — no CUDA ─────────────────────────────────────────────────────
DEVICE = torch.device('cpu')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 15
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Tunable experiment settings ──────────────────────────────────────────────
USE_SMALL_SAMPLE          = False
SMALL_SAMPLE_SIZE         = 600    # shared across both label sets
BATCH_SIZE                = 16
EPOCHS                    = 15
LR                        = 1e-4
WEIGHT_DECAY              = 1e-4
D_MODEL                   = 256
N_HEADS                   = 4     # must divide D_MODEL evenly
DROPOUT                   = 0.2
FREEZE_IMAGE_BACKBONE     = True
EARLY_STOPPING_PATIENCE   = 5
BERT_MODEL_NAME           = 'distilbert-base-uncased'

# ── Column names ─────────────────────────────────────────────────────────────
TEXT_COLUMN       = 'tweet_text'
IMAGE_PATH_COLUMN = 'image_path'
OLD_LABEL_COL     = 'text_human'   # original human annotations
NEW_LABEL_COL     = 'text_llm'     # updated LLM annotations (Phase 2 output)

# ── Project paths ─────────────────────────────────────────────────────────────
def find_project_root(marker: str = 'CrisisMMD_v2.0') -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"Cannot find '{marker}' from cwd={Path.cwd()}")

PROJECT_ROOT    = find_project_root()
CRISIS_ROOT     = PROJECT_ROOT / 'CrisisMMD_v2.0'
UPDATED_ANN_DIR = PROJECT_ROOT / 'CrisisMMD_v2_Annotations'
CACHE_DIR       = PROJECT_ROOT / 'cache'
RESULTS_DIR     = PROJECT_ROOT / 'thesis_analysis_results' / 'phase3_annotation_benchmark'
CACHE_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device          : {DEVICE}')
print(f'Project root    : {PROJECT_ROOT}')
print(f'Updated ann dir : {UPDATED_ANN_DIR}')
print(f'BERT model      : {BERT_MODEL_NAME}')
print(f'D_MODEL={D_MODEL}  N_HEADS={N_HEADS}  DROPOUT={DROPOUT}  EPOCHS={EPOCHS}')
print(f'USE_SMALL_SAMPLE={USE_SMALL_SAMPLE}  SMALL_SAMPLE_SIZE={SMALL_SAMPLE_SIZE}')


c:\Users\Kaleb\Desktop\Spring 2026\Data Mining\Project\crisisMMDDataSet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device          : cpu
Project root    : C:\Users\Kaleb\Desktop\Spring 2026\Data Mining\Project\crisisMMDDataSet
Updated ann dir : C:\Users\Kaleb\Desktop\Spring 2026\Data Mining\Project\crisisMMDDataSet\CrisisMMD_v2_Annotations
BERT model      : distilbert-base-uncased
D_MODEL=256  N_HEADS=4  DROPOUT=0.2  EPOCHS=15
USE_SMALL_SAMPLE=False  SMALL_SAMPLE_SIZE=600


In [2]:
# ── Cell 2: Load and align annotations ───────────────────────────────────────
# The updated annotation CSVs already contain BOTH old (text_human) and
# new (text_llm) labels per sample → single source of truth.

all_dfs = []
for event_dir in sorted(UPDATED_ANN_DIR.iterdir()):
    if not event_dir.is_dir():
        continue
    for csv_file in sorted(event_dir.glob('*.csv')):
        df_tmp = pd.read_csv(csv_file, low_memory=False)
        all_dfs.append(df_tmp)

if not all_dfs:
    raise RuntimeError(f'No CSV files found under {UPDATED_ANN_DIR}')

master_df = pd.concat(all_dfs, ignore_index=True)

# Resolve full image paths
master_df['full_image_path'] = master_df[IMAGE_PATH_COLUMN].apply(
    lambda p: CRISIS_ROOT / str(p)
)

# Keep only rows that have both label columns, tweet text, and a real image
required = [OLD_LABEL_COL, NEW_LABEL_COL, TEXT_COLUMN, IMAGE_PATH_COLUMN]
master_df = master_df.dropna(subset=required).copy()
master_df = master_df[master_df['full_image_path'].apply(lambda p: p.exists())]
master_df = master_df.reset_index(drop=True)

print(f'Total usable rows: {len(master_df)}')
print(f'\nOld label distribution ({OLD_LABEL_COL}):')
print(master_df[OLD_LABEL_COL].value_counts().to_string())
print(f'\nNew label distribution ({NEW_LABEL_COL}):')
print(master_df[NEW_LABEL_COL].value_counts().to_string())

# ── Optional small-sample for faster runs ────────────────────────────────────
if USE_SMALL_SAMPLE and len(master_df) > SMALL_SAMPLE_SIZE:
    strat = master_df[OLD_LABEL_COL] if master_df[OLD_LABEL_COL].value_counts().min() >= 2 else None
    master_df, _ = train_test_split(
        master_df, train_size=SMALL_SAMPLE_SIZE, random_state=SEED, stratify=strat
    )
    master_df = master_df.reset_index(drop=True)
    print(f'\nUsing sampled rows: {len(master_df)}')

# ── Shared splits (same rows for BOTH label sets — ensures fair comparison) ──
def can_stratify(labels: pd.Series, test_size: float) -> bool:
    counts = labels.value_counts()
    n = len(labels)
    n_test = max(1, int(np.ceil(n * test_size)))
    return counts.min() >= 2 and (n - n_test) >= len(counts) and n_test >= len(counts)

strat1 = master_df[OLD_LABEL_COL] if can_stratify(master_df[OLD_LABEL_COL], 0.3) else None
train_df, temp_df = train_test_split(
    master_df, test_size=0.3, random_state=SEED, stratify=strat1
)
strat2 = temp_df[OLD_LABEL_COL] if can_stratify(temp_df[OLD_LABEL_COL], 0.5) else None
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=strat2
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'\nShared split sizes → Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')


Total usable rows: 18082

Old label distribution (text_human):
text_human
other_relevant_information                6499
not_humanitarian                          5227
rescue_volunteering_or_donation_effort    3776
infrastructure_and_utility_damage         1428
injured_or_dead_people                     533
affected_individuals                       517
vehicle_damage                              61
missing_or_found_people                     41

New label distribution (text_llm):
text_llm
other_relevant_information                5933
rescue_volunteering_or_donation_effort    3637
infrastructure_and_utility_damage         2771
affected_individuals                      2703
not_humanitarian                          2489
injured_or_dead_people                     340
vehicle_damage                             113
missing_or_found_people                     96

Shared split sizes → Train: 12657, Val: 2712, Test: 2713


In [3]:
# ── Cell 3: DistilBERT text embeddings (CPU, frozen, cached) ─────────────────
from transformers import AutoTokenizer, AutoModel

BERT_CACHE = CACHE_DIR / f'bert_cls_{BERT_MODEL_NAME.replace("/", "_")}.pkl'

_tokenizer  = None
_bert_model = None

def _load_bert():
    global _tokenizer, _bert_model
    if _tokenizer is None:
        print(f'Loading {BERT_MODEL_NAME} ...')
        _tokenizer  = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
        _bert_model = AutoModel.from_pretrained(BERT_MODEL_NAME)
        _bert_model.eval()
        _bert_model.to(DEVICE)

@torch.no_grad()
def _encode_batch(texts: list, batch_size: int = 32) -> np.ndarray:
    _load_bert()
    results = []
    for i in tqdm(range(0, len(texts), batch_size), desc='BERT encoding'):
        batch = texts[i : i + batch_size]
        enc = _tokenizer(
            batch, padding=True, truncation=True,
            max_length=128, return_tensors='pt'
        ).to(DEVICE)
        out = _bert_model(**enc)
        cls = out.last_hidden_state[:, 0, :].cpu().numpy()
        results.append(cls)
    return np.vstack(results).astype(np.float32)

# Load or build the per-text embedding cache
if BERT_CACHE.exists():
    with open(BERT_CACHE, 'rb') as f:
        bert_cache: dict = pickle.load(f)
    print(f'Loaded BERT cache: {len(bert_cache)} entries')
else:
    bert_cache = {}

all_texts = (
    pd.concat([train_df, val_df, test_df])[TEXT_COLUMN]
    .fillna('').astype(str).tolist()
)
missing = list({t for t in all_texts if t not in bert_cache})

if missing:
    print(f'Encoding {len(missing)} new texts with BERT ...')
    embs = _encode_batch(missing)
    for txt, emb in zip(missing, embs):
        bert_cache[txt] = emb
    with open(BERT_CACHE, 'wb') as f:
        pickle.dump(bert_cache, f)
    print('BERT cache saved.')
else:
    print('All texts already in cache — skipping encoding.')

def get_bert_embeddings(df: pd.DataFrame) -> np.ndarray:
    texts = df[TEXT_COLUMN].fillna('').astype(str).tolist()
    return np.stack([bert_cache[t] for t in texts]).astype(np.float32)

X_text_train = get_bert_embeddings(train_df)
X_text_val   = get_bert_embeddings(val_df)
X_text_test  = get_bert_embeddings(test_df)
TEXT_DIM     = X_text_train.shape[1]

print(f'BERT embedding dim : {TEXT_DIM}')
print(f'Train: {X_text_train.shape}  Val: {X_text_val.shape}  Test: {X_text_test.shape}')


Loaded BERT cache: 2964 entries
Encoding 13094 new texts with BERT ...
Loading distilbert-base-uncased ...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8926.90it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BERT encoding: 100%|██████████| 410/410 [01:39<00:00,  4.11it/s]


BERT cache saved.
BERT embedding dim : 768
Train: (12657, 768)  Val: (2712, 768)  Test: (2713, 768)


In [4]:
# ── Cell 4: Model architecture + hyperparameter configurations ────────────────
# All configs: DistilBERT text + DenseNet121 image → MultiheadAttention Fusion
weights         = DenseNet121_Weights.IMAGENET1K_V1
image_transform = weights.transforms()


class CrisisDataset(Dataset):
    """Always loads both image and text — used by every config."""
    def __init__(self, df: pd.DataFrame, text_embeddings: np.ndarray,
                 labels: np.ndarray):
        self.df              = df.reset_index(drop=True)
        self.text_embeddings = text_embeddings
        self.labels          = labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        image    = Image.open(row['full_image_path']).convert('RGB')
        image    = image_transform(image)
        text_vec = torch.tensor(self.text_embeddings[idx], dtype=torch.float32)
        label    = torch.tensor(int(self.labels[idx]),     dtype=torch.long)
        return image, text_vec, label


class MultimodalFusion(nn.Module):
    """DistilBERT CLS token + DenseNet121 features → MultiheadAttention → MLP."""

    def __init__(self, text_dim: int, num_classes: int,
                 d_model: int, n_heads: int, dropout: float,
                 freeze_image: bool = True):
        super().__init__()
        backbone           = models.densenet121(weights=weights)
        self.image_encoder = backbone.features
        self.image_pool    = nn.AdaptiveAvgPool2d((1, 1))
        if freeze_image:
            for p in self.image_encoder.parameters():
                p.requires_grad = False
        self.image_proj = nn.Linear(1024,     d_model)
        self.text_proj  = nn.Linear(text_dim, d_model)
        self.attn       = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.norm       = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes),
        )

    def forward(self, images, text_vecs):
        img_feat    = self.image_pool(self.image_encoder(images)).flatten(1)
        img_tok     = self.image_proj(img_feat)
        txt_tok     = self.text_proj(text_vecs)
        tokens      = torch.stack([txt_tok, img_tok], dim=1)   # (B, 2, d_model)
        attn_out, _ = self.attn(tokens, tokens, tokens)
        fused       = self.norm(tokens + attn_out).mean(dim=1)  # mean-pool 2 tokens
        return self.classifier(fused)


# ── Hyperparameter configurations ─────────────────────────────────────────────
# n_heads must evenly divide d_model
CONFIGS = {
    'A — Baseline':           dict(d_model=256, lr=1e-4,  dropout=0.20, n_heads=4, freeze_image=True),
    'B — Larger Model':       dict(d_model=512, lr=1e-4,  dropout=0.20, n_heads=8, freeze_image=True),
    'C — Unfrozen Backbone':  dict(d_model=256, lr=1e-4,  dropout=0.30, n_heads=4, freeze_image=False),
    'D — Larger + Unfrozen':  dict(d_model=512, lr=3e-5,  dropout=0.30, n_heads=8, freeze_image=False),
}

print('Architecture : DistilBERT + DenseNet121 (Cross-Modal Attention Fusion)')
print(f'{"─"*65}')
print(f'{"Config":<35} {"d_model":>7} {"lr":>7} {"dropout":>8} {"freeze":>7}')
print(f'{"─"*65}')
for name, hp in CONFIGS.items():
    print(f'{name:<35} {hp["d_model"]:>7} {hp["lr"]:>7} {hp["dropout"]:>8} {str(hp["freeze_image"]):>7}')
print(f'{"─"*65}')


Architecture : DistilBERT + DenseNet121 (Cross-Modal Attention Fusion)
─────────────────────────────────────────────────────────────────
Config                              d_model      lr  dropout  freeze
─────────────────────────────────────────────────────────────────
A — Baseline                            256  0.0001      0.2    True
B — Larger Model                        512  0.0001      0.2    True
C — Unfrozen Backbone                   256  0.0001      0.3   False
D — Larger + Unfrozen                   512   3e-05      0.3   False
─────────────────────────────────────────────────────────────────


In [5]:
# ── Cell 4.5: Sanity check — verify full pipeline before the benchmark ────────
# Runs 2 batches through Config A (fastest) on OLD label set.
# Checks: loss is finite, logits shape is correct, backward pass completes.
# If anything is wrong this cell raises immediately so you don't waste time.

import time as _time

print('=' * 60)
print('SANITY CHECK — verifying pipeline before full benchmark')
print('=' * 60)

_le_sanity = LabelEncoder()
_sanity_labels = pd.concat([
    train_df[OLD_LABEL_COL], val_df[OLD_LABEL_COL], test_df[OLD_LABEL_COL]
]).astype(str)
_le_sanity.fit(_sanity_labels)
_nc   = len(_le_sanity.classes_)
_y_tr = _le_sanity.transform(train_df[OLD_LABEL_COL].astype(str))
_y_vl = _le_sanity.transform(val_df[OLD_LABEL_COL].astype(str))
_y_te = _le_sanity.transform(test_df[OLD_LABEL_COL].astype(str))

print(f'  Label set      : {OLD_LABEL_COL}')
print(f'  Num classes    : {_nc}  {_le_sanity.classes_.tolist()}')

# Build a single loader, grab 2 batches
_loader = DataLoader(
    CrisisDataset(train_df, X_text_train, _y_tr),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

_cfg_a  = CONFIGS['A — Baseline']
_model  = MultimodalFusion(
    text_dim    = TEXT_DIM,
    num_classes = _nc,
    d_model     = _cfg_a['d_model'],
    n_heads     = _cfg_a['n_heads'],
    dropout     = _cfg_a['dropout'],
    freeze_image= _cfg_a['freeze_image'],
).to(DEVICE)

_crit = nn.CrossEntropyLoss()
_opt  = optim.AdamW(_model.parameters(), lr=_cfg_a['lr'])
_model.train()

_issues = []
_t0     = _time.time()

for _batch_idx, (imgs, txts, labs) in enumerate(_loader):
    if _batch_idx >= 2:
        break
    imgs, txts, labs = imgs.to(DEVICE), txts.to(DEVICE), labs.to(DEVICE)
    _opt.zero_grad()
    _logits = _model(imgs, txts)
    _loss   = _crit(_logits, labs)
    _loss.backward()
    _opt.step()

    expected_shape = (imgs.size(0), _nc)
    actual_shape   = tuple(_logits.shape)
    loss_val       = _loss.item()

    print(f'  Batch {_batch_idx + 1}/2 | '
          f'logits shape={actual_shape} (expected {expected_shape}) | '
          f'loss={loss_val:.4f} | finite={torch.isfinite(_loss).item()}')

    if actual_shape != expected_shape:
        _issues.append(f'Batch {_batch_idx+1}: logits shape {actual_shape} != {expected_shape}')
    if not torch.isfinite(_loss):
        _issues.append(f'Batch {_batch_idx+1}: loss is not finite ({loss_val})')

del _model, _crit, _opt, _loader, _logits, _loss

_elapsed = _time.time() - _t0
print(f'{"─"*60}')
if _issues:
    for issue in _issues:
        print(f'  ✗ {issue}')
    raise RuntimeError(f'Sanity check FAILED — fix the issues above before running Cell 6.')
else:
    print(f'  ✓ SANITY CHECK PASSED  ({_elapsed:.1f}s)')
    print(f'  Ready to run 8 full experiments (4 configs × 2 annotation sets).')
print('=' * 60)


SANITY CHECK — verifying pipeline before full benchmark
  Label set      : text_human
  Num classes    : 8  ['affected_individuals', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'missing_or_found_people', 'not_humanitarian', 'other_relevant_information', 'rescue_volunteering_or_donation_effort', 'vehicle_damage']
  Batch 1/2 | logits shape=(16, 8) (expected (16, 8)) | loss=2.0155 | finite=True
  Batch 2/2 | logits shape=(16, 8) (expected (16, 8)) | loss=1.9818 | finite=True
────────────────────────────────────────────────────────────
  ✓ SANITY CHECK PASSED  (1.1s)
  Ready to run 8 full experiments (4 configs × 2 annotation sets).


In [6]:
# ── Cell 5: Training and evaluation utilities ─────────────────────────────────
import time


def make_loaders(y_train: np.ndarray, y_val: np.ndarray, y_test: np.ndarray,
                 tr_df=None, vl_df=None, te_df=None,
                 x_tr=None, x_vl=None, x_te=None):
    """Build DataLoaders. Defaults to the global humanitarian splits."""
    tr_df = tr_df if tr_df is not None else train_df
    vl_df = vl_df if vl_df is not None else val_df
    te_df = te_df if te_df is not None else test_df
    x_tr  = x_tr  if x_tr  is not None else X_text_train
    x_vl  = x_vl  if x_vl  is not None else X_text_val
    x_te  = x_te  if x_te  is not None else X_text_test
    kw = dict(batch_size=BATCH_SIZE, num_workers=0, pin_memory=False)
    return (
        DataLoader(CrisisDataset(tr_df, x_tr, y_train), shuffle=True,  **kw),
        DataLoader(CrisisDataset(vl_df, x_vl, y_val),   shuffle=False, **kw),
        DataLoader(CrisisDataset(te_df, x_te, y_test),  shuffle=False, **kw),
    )


def class_weights_tensor(labels: np.ndarray, num_classes: int) -> torch.Tensor:
    counts = pd.Series(labels).value_counts().sort_index()
    n      = len(labels)
    w      = [n / (num_classes * counts.get(i, 1)) for i in range(num_classes)]
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def run_epoch(model, loader, criterion, optimizer=None, train_mode: bool = True):
    model.train(train_mode)
    total_loss, y_true, y_pred = 0.0, [], []
    for images, text_vecs, labels in loader:
        images, text_vecs, labels = (
            images.to(DEVICE), text_vecs.to(DEVICE), labels.to(DEVICE)
        )
        with torch.set_grad_enabled(train_mode):
            logits = model(images, text_vecs)
            loss   = criterion(logits, labels)
            if train_mode and optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(logits.argmax(1).cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    wf1      = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    acc      = accuracy_score(y_true, y_pred)
    return avg_loss, wf1, acc


def train_and_evaluate(model, train_loader, val_loader, test_loader,
                       y_train: np.ndarray, num_classes: int, lr: float):
    """Train for up to EPOCHS epochs with early stopping.
    Returns (metrics_dict, y_true_list, y_pred_list) for per-event breakdown."""
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor(y_train, num_classes))
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    best_val_wf1, best_state, no_improve = -1.0, None, 0

    for epoch in range(1, EPOCHS + 1):
        ep_start = time.time()
        tr_loss, tr_wf1, tr_acc = run_epoch(model, train_loader, criterion, optimizer, True)
        vl_loss, vl_wf1, vl_acc = run_epoch(model, val_loader,   criterion, None,      False)
        ep_sec = time.time() - ep_start

        print(
            f'    Epoch {epoch}/{EPOCHS} ({ep_sec:.0f}s) | '
            f'train  loss={tr_loss:.4f}  acc={tr_acc:.4f} | '
            f'val    loss={vl_loss:.4f}  acc={vl_acc:.4f}  wF1={vl_wf1:.4f}'
        )

        if vl_wf1 > best_val_wf1:
            best_val_wf1 = vl_wf1
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve   = 0
            print(f'    ↑ New best val wF1={best_val_wf1:.4f} — checkpoint saved')
        else:
            no_improve += 1
            if EARLY_STOPPING_PATIENCE > 0 and no_improve >= EARLY_STOPPING_PATIENCE:
                print(f'    ⏹ Early stopping triggered (no improvement for {no_improve} epochs)')
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # ── Test evaluation ───────────────────────────────────────────────────────
    model.eval()
    y_true_t, y_pred_t = [], []
    with torch.no_grad():
        for images, text_vecs, labels in test_loader:
            images, text_vecs = images.to(DEVICE), text_vecs.to(DEVICE)
            preds = model(images, text_vecs).argmax(1)
            y_true_t.extend(labels.numpy())
            y_pred_t.extend(preds.cpu().numpy())

    test_acc = accuracy_score(y_true_t, y_pred_t)
    test_wf1 = f1_score(y_true_t, y_pred_t, average='weighted', zero_division=0)
    test_mf1 = f1_score(y_true_t, y_pred_t, average='macro',    zero_division=0)

    print(f'    ── TEST  acc={test_acc:.4f}  wF1={test_wf1:.4f}  mF1={test_mf1:.4f}')

    metrics = {
        'test_accuracy': test_acc,
        'test_wF1':      test_wf1,
        'test_mF1':      test_mf1,
        'best_val_wF1':  best_val_wf1,
    }
    return metrics, y_true_t, y_pred_t

print('Training utilities ready.')


Training utilities ready.


In [ ]:
# ── Cell 6: Run all experiments ───────────────────────────────────────────────
# 4 hyperparameter configs × 2 annotation sets = 8 training runs (15 epochs each)

import time

LABEL_SETS = {
    'Old Annotations (text_human)': OLD_LABEL_COL,
    'New Annotations (text_llm)':   NEW_LABEL_COL,
}

TOTAL_RUNS       = len(CONFIGS) * len(LABEL_SETS)
run_counter      = 0
all_results      = []
all_event_results = []          # NEW: per-event breakdown
benchmark_start  = time.time()

for label_name, label_col in LABEL_SETS.items():
    # Encode labels for this annotation set
    le = LabelEncoder()
    le.fit(
        pd.concat([train_df[label_col], val_df[label_col], test_df[label_col]])
        .astype(str)
    )
    num_classes = len(le.classes_)
    y_train = le.transform(train_df[label_col].astype(str))
    y_val   = le.transform(val_df[label_col].astype(str))
    y_test  = le.transform(test_df[label_col].astype(str))

    # Shared loaders for all 4 configs within this label set
    train_loader, val_loader, test_loader = make_loaders(y_train, y_val, y_test)

    print(f'\n{"#"*65}')
    print(f'  Annotation Set : {label_name}')
    print(f'  Classes ({num_classes})   : {le.classes_.tolist()}')
    print(f'{"#"*65}')

    for cfg_name, hp in CONFIGS.items():
        run_counter += 1
        run_start = time.time()

        print(f'\n  ┌─ [Run {run_counter}/{TOTAL_RUNS}] {cfg_name}')
        print(f'  │  d_model={hp["d_model"]}  lr={hp["lr"]}  '
              f'dropout={hp["dropout"]}  freeze={hp["freeze_image"]}  '
              f'epochs={EPOCHS}')
        print(f'  │')

        model = MultimodalFusion(
            text_dim     = TEXT_DIM,
            num_classes  = num_classes,
            d_model      = hp['d_model'],
            n_heads      = hp['n_heads'],
            dropout      = hp['dropout'],
            freeze_image = hp['freeze_image'],
        ).to(DEVICE)

        metrics, y_true_t, y_pred_t = train_and_evaluate(
            model, train_loader, val_loader, test_loader,
            y_train, num_classes, lr=hp['lr']
        )

        run_elapsed = time.time() - run_start
        all_results.append({
            'Annotation Set': label_name,
            'Model Config':   cfg_name,
            'D_MODEL':        hp['d_model'],
            'LR':             hp['lr'],
            'Dropout':        hp['dropout'],
            'Frozen':         hp['freeze_image'],
            **{k: round(v, 4) for k, v in metrics.items()},
        })

        # ── Per-event breakdown ────────────────────────────────────────────────
        test_events = test_df['disaster_type'].fillna('unknown').values
        for event in sorted(set(test_events)):
            mask = test_events == event
            n    = int(mask.sum())
            if n < 5:
                continue
            yt = [y_true_t[i] for i, m in enumerate(mask) if m]
            yp = [y_pred_t[i] for i, m in enumerate(mask) if m]
            all_event_results.append({
                'Event':          event,
                'Annotation Set': label_name,
                'Model Config':   cfg_name,
                'n_test':         n,
                'test_accuracy':  round(accuracy_score(yt, yp), 4),
                'test_wF1':       round(f1_score(yt, yp, average='weighted', zero_division=0), 4),
                'test_mF1':       round(f1_score(yt, yp, average='macro',    zero_division=0), 4),
            })

        print(f'  └─ Done in {run_elapsed:.0f}s  '
              f'[Acc={metrics["test_accuracy"]:.4f}  '
              f'wF1={metrics["test_wF1"]:.4f}  '
              f'mF1={metrics["test_mF1"]:.4f}]')
        del model

# ── Summary table ─────────────────────────────────────────────────────────────
total_elapsed    = time.time() - benchmark_start
results_df       = pd.DataFrame(all_results)
event_results_df = pd.DataFrame(all_event_results)

print(f'\n\n{"="*65}')
print(f'ALL {TOTAL_RUNS} RUNS COMPLETE  (total: {total_elapsed/60:.1f} min)')
print(f'{"="*65}')
print(results_df[['Annotation Set', 'Model Config',
                   'test_accuracy', 'test_wF1', 'test_mF1']].to_string(index=False))

if not event_results_df.empty:
    print(f'\nPer-event records collected: {len(event_results_df)} '
          f'({event_results_df["Event"].nunique()} events × configs × annotation sets)')



#################################################################
  Annotation Set : Old Annotations (text_human)
  Classes (8)   : ['affected_individuals', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'missing_or_found_people', 'not_humanitarian', 'other_relevant_information', 'rescue_volunteering_or_donation_effort', 'vehicle_damage']
#################################################################

  ┌─ [Run 1/8] A — Baseline
  │  d_model=256  lr=0.0001  dropout=0.2  freeze=True  epochs=15
  │
    Epoch 1/15 (423s) | train  loss=1.6016  acc=0.4865 | val    loss=1.3549  acc=0.6029  wF1=0.5876
    ↑ New best val wF1=0.5876 — checkpoint saved
    Epoch 2/15 (422s) | train  loss=1.2250  acc=0.5789 | val    loss=1.2101  acc=0.6055  wF1=0.6056
    ↑ New best val wF1=0.6056 — checkpoint saved
    Epoch 3/15 (423s) | train  loss=1.0851  acc=0.6030 | val    loss=1.1603  acc=0.6217  wF1=0.6185
    ↑ New best val wF1=0.6185 — checkpoint saved
    Epoch 4/15 (420s) | train  

In [ ]:
# ── Cell 7: Comparison table and visualisation ────────────────────────────────

OLD_SET = 'Old Annotations (text_human)'
NEW_SET = 'New Annotations (text_llm)'
METRICS = {'test_accuracy': 'Accuracy', 'test_wF1': 'Weighted F1', 'test_mF1': 'Macro F1'}

# ── Pivoted per-metric comparison tables ──────────────────────────────────────
print('=' * 80)
print('PHASE 3 RESULTS — Old vs New Annotations')
print('=' * 80)

for col, label in METRICS.items():
    pivot = results_df.pivot(index='Model Config', columns='Annotation Set', values=col)
    pivot.columns.name = None
    pivot.index.name   = None
    print(f'\n── {label} ─────────────────────────────────────────────────────────')
    print(pivot.round(4).to_string())

# ── Delta table ───────────────────────────────────────────────────────────────
print('\n' + '=' * 80)
print('DELTA:  New – Old   (positive = new annotations helped)')
print('=' * 80)

delta_rows = []
for cfg in results_df['Model Config'].unique():
    sub = results_df[results_df['Model Config'] == cfg].set_index('Annotation Set')
    if OLD_SET in sub.index and NEW_SET in sub.index:
        delta_rows.append({
            'Model Config':  cfg,
            'Δ Accuracy':    round(sub.loc[NEW_SET, 'test_accuracy'] - sub.loc[OLD_SET, 'test_accuracy'], 4),
            'Δ Weighted F1': round(sub.loc[NEW_SET, 'test_wF1']      - sub.loc[OLD_SET, 'test_wF1'],      4),
            'Δ Macro F1':    round(sub.loc[NEW_SET, 'test_mF1']      - sub.loc[OLD_SET, 'test_mF1'],      4),
        })

delta_df = pd.DataFrame(delta_rows)
print(delta_df.to_string(index=False))

# ── Bar chart ─────────────────────────────────────────────────────────────────
sns.set_style('whitegrid')
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

palette = {
    OLD_SET: '#4472C4',
    NEW_SET: '#ED7D31',
}

for ax, (metric_col, metric_label) in zip(axes, METRICS.items()):
    pivot = results_df.pivot(
        index='Model Config', columns='Annotation Set', values=metric_col
    )
    x     = np.arange(len(pivot))
    width = 0.35

    for j, (ann_set, color) in enumerate(palette.items()):
        if ann_set not in pivot.columns:
            continue
        bars = ax.bar(
            x + (j - 0.5) * width, pivot[ann_set], width,
            label=ann_set, color=color, alpha=0.88, edgecolor='white'
        )
        for bar in bars:
            h = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold'
            )

    ax.set_title(metric_label, fontsize=12, fontweight='bold', pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [c.replace(' — ', '\n') for c in pivot.index],
        rotation=20, ha='right', fontsize=8
    )
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(metric_label)
    ax.legend(fontsize=8, loc='lower right')

fig.suptitle(
    'Humanitarian Task: Old vs New Annotations — Model Performance Comparison',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
out_path = RESULTS_DIR / 'old_vs_new_comparison.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\nChart saved  → {out_path}')

# ── Save results to CSV ───────────────────────────────────────────────────────
csv_path = RESULTS_DIR / 'phase3_benchmark_results.csv'
results_df.to_csv(csv_path, index=False)
print(f'Results CSV  → {csv_path}')


PHASE 3 RESULTS — Old vs New Annotations

── Accuracy ─────────────────────────────────────────────────────────
                       New Annotations (text_llm)  Old Annotations (text_human)
A — Baseline                               0.5333                        0.4667
B — Larger Model                           0.6000                        0.4111
C — Unfrozen Backbone                      0.4333                        0.4222
D — Larger + Unfrozen                      0.4667                        0.4778

── Weighted F1 ─────────────────────────────────────────────────────────
                       New Annotations (text_llm)  Old Annotations (text_human)
A — Baseline                               0.5235                        0.4732
B — Larger Model                           0.5875                        0.4135
C — Unfrozen Backbone                      0.4059                        0.4058
D — Larger + Unfrozen                      0.4486                        0.4657

── Macro F1 ─

## Per-Event Performance Breakdown — Humanitarian Task

Each disaster event is evaluated separately on the same trained models.  
Metric values are **averaged across the 4 hyperparameter configs**.  
Events with fewer than 5 test samples are excluded from reporting.


In [ ]:
# ── Cell 8: Per-event tables, charts, and standalone files ───────────────────
# Uses event_results_df built in Cell 6 (per-run per-event predictions).

if event_results_df.empty:
    print('No per-event data available (all events had < 5 test samples).')
else:
    _OLD = 'Old Annotations (text_human)'
    _NEW = 'New Annotations (text_llm)'
    EV_METRICS = {'test_accuracy': 'Accuracy', 'test_wF1': 'Weighted F1', 'test_mF1': 'Macro F1'}
    _events = sorted(event_results_df['Event'].unique())

    # ── 1. Summary tables: avg across configs per event ───────────────────────
    ev_avg = (
        event_results_df
        .groupby(['Event', 'Annotation Set'])[list(EV_METRICS.keys())]
        .mean()
        .round(4)
        .reset_index()
    )

    print('=' * 80)
    print('PER-EVENT RESULTS — Humanitarian Task  (averaged across 4 configs)')
    print('=' * 80)
    for metric_col, metric_label in EV_METRICS.items():
        pivot = ev_avg.pivot(index='Event', columns='Annotation Set', values=metric_col)
        pivot.columns.name = None
        pivot.index.name   = None
        if _OLD in pivot.columns and _NEW in pivot.columns:
            pivot['Δ (New−Old)'] = (pivot[_NEW] - pivot[_OLD]).round(4)
        print(f'\n── {metric_label} ──────────────────────────────────────────────')
        print(pivot.round(4).to_string())

    # ── 2. Per-event delta table: event × config ──────────────────────────────
    print('\n' + '=' * 80)
    print('PER-EVENT × CONFIG DELTA — Accuracy  (New − Old)')
    print('=' * 80)
    ev_delta_rows = []
    for event in _events:
        row = {'Event': event}
        for cfg in CONFIGS.keys():
            sub = event_results_df[
                (event_results_df['Event'] == event) &
                (event_results_df['Model Config'] == cfg)
            ].set_index('Annotation Set')
            if _OLD in sub.index and _NEW in sub.index:
                row[cfg] = round(sub.loc[_NEW, 'test_accuracy'] - sub.loc[_OLD, 'test_accuracy'], 4)
            else:
                row[cfg] = float('nan')
        ev_delta_rows.append(row)
    ev_delta_df = pd.DataFrame(ev_delta_rows).set_index('Event')
    print(ev_delta_df.to_string())

    # ── 3. Bar chart: per-event avg old vs new (3 metrics) ────────────────────
    palette_ev = {_OLD: '#4472C4', _NEW: '#ED7D31'}
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for ax, (metric_col, metric_label) in zip(axes, EV_METRICS.items()):
        pivot = ev_avg.pivot(index='Event', columns='Annotation Set', values=metric_col)
        x     = np.arange(len(pivot))
        width = 0.35
        for j, (ann_set, color) in enumerate(palette_ev.items()):
            if ann_set not in pivot.columns:
                continue
            bars = ax.bar(x + (j - 0.5) * width, pivot[ann_set], width,
                          label=ann_set, color=color, alpha=0.88, edgecolor='white')
            for bar in bars:
                h = bar.get_height()
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                        f'{h:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
        ax.set_title(metric_label, fontsize=12, fontweight='bold', pad=10)
        ax.set_xticks(x)
        ax.set_xticklabels([e.replace('_', '\n') for e in pivot.index],
                           rotation=20, ha='right', fontsize=8)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel(metric_label)
        ax.legend(fontsize=8, loc='lower right')
    fig.suptitle(
        'Humanitarian Task: Per-Event Performance — Old vs New Annotations (Avg. across 4 Configs)',
        fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    ev_bar_path = RESULTS_DIR / 'per_event_humanitarian_comparison.png'
    plt.savefig(ev_bar_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nBar chart saved → {ev_bar_path}')

    # ── 4. Heatmap: delta by event × config (3 metrics) ───────────────────────
    fig2, axes2 = plt.subplots(1, 3, figsize=(20, max(4, len(_events) * 0.9 + 1)))
    for ax, (metric_col, metric_label) in zip(axes2, EV_METRICS.items()):
        heat_data = {}
        for cfg in CONFIGS.keys():
            col_vals = []
            for event in _events:
                sub = event_results_df[
                    (event_results_df['Event'] == event) &
                    (event_results_df['Model Config'] == cfg)
                ].set_index('Annotation Set')
                if _OLD in sub.index and _NEW in sub.index:
                    col_vals.append(round(sub.loc[_NEW, metric_col] - sub.loc[_OLD, metric_col], 4))
                else:
                    col_vals.append(float('nan'))
            heat_data[cfg.replace(' — ', '\n')] = col_vals
        heat_df = pd.DataFrame(heat_data, index=[e.replace('_', '\n') for e in _events])
        sns.heatmap(heat_df, ax=ax, annot=True, fmt='.3f', center=0,
                    cmap='RdYlGn', linewidths=0.5, linecolor='grey',
                    cbar_kws={'label': 'Δ (New−Old)'})
        ax.set_title(f'Δ {metric_label} (New − Old)', fontsize=11, fontweight='bold')
        ax.set_xlabel('')
    fig2.suptitle('Humanitarian Task: Per-Event Annotation Delta — New vs Old (Δ = New − Old)',
                  fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    ev_heat_path = RESULTS_DIR / 'per_event_humanitarian_delta_heatmap.png'
    plt.savefig(ev_heat_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Heatmap saved     → {ev_heat_path}')

    # ── 5. Save CSVs ──────────────────────────────────────────────────────────
    ev_full_csv = RESULTS_DIR / 'per_event_humanitarian_results.csv'
    event_results_df.to_csv(ev_full_csv, index=False)
    print(f'Full per-event CSV → {ev_full_csv}')

    ev_avg_csv = RESULTS_DIR / 'per_event_humanitarian_avg.csv'
    ev_avg.to_csv(ev_avg_csv, index=False)
    print(f'Avg per-event CSV  → {ev_avg_csv}')

    ev_delta_csv = RESULTS_DIR / 'per_event_humanitarian_delta.csv'
    ev_delta_df.to_csv(ev_delta_csv)
    print(f'Delta CSV          → {ev_delta_csv}')


PER-EVENT RESULTS — Humanitarian Task  (averaged across 4 configs)

── Accuracy ──────────────────────────────────────────────
                      New Annotations (text_llm)  Old Annotations (text_human)  Δ (New−Old)
california_wildfires                      0.3750                        0.5000      -0.1250
hurricane_harvey                          0.4167                        0.4643      -0.0476
hurricane_irma                            0.5139                        0.3889       0.1250
hurricane_maria                           0.5866                        0.5096       0.0770

── Weighted F1 ──────────────────────────────────────────────
                      New Annotations (text_llm)  Old Annotations (text_human)  Δ (New−Old)
california_wildfires                      0.3631                        0.4945      -0.1314
hurricane_harvey                          0.3994                        0.4362      -0.0368
hurricane_irma                            0.5106                        0.

## Damage Severity Task Benchmark

Compares model performance on **image damage severity classification** using:
- `image_damage` — original human annotations  
- `image_damage_llm` — updated LLM annotations (Phase 2)

Classes: `severe_damage`, `mild_damage`, `little_or_no_damage`, `dont_know_or_cant_judge`  
Only rows with **both** human and LLM damage labels are used (~2,766 rows).  
Same 4 configs, 15 epochs, early stopping patience=5, CPU only.


In [21]:
# ── Cell 9: Damage severity — data prep & benchmark ──────────────────────────
import time

DMG_OLD_COL = 'image_damage'
DMG_NEW_COL = 'image_damage_llm'

# ── Reload full dataset filtered to rows with both damage labels ──────────────
_dmg_dfs = []
for _ev_dir in sorted(UPDATED_ANN_DIR.iterdir()):
    if not _ev_dir.is_dir():
        continue
    for _csv in sorted(_ev_dir.glob('*.csv')):
        _dmg_dfs.append(pd.read_csv(_csv, low_memory=False))

_dmg_full = pd.concat(_dmg_dfs, ignore_index=True)
_dmg_full['full_image_path'] = _dmg_full[IMAGE_PATH_COLUMN].apply(
    lambda p: CRISIS_ROOT / str(p)
)
_dmg_required = [DMG_OLD_COL, DMG_NEW_COL, TEXT_COLUMN, IMAGE_PATH_COLUMN]
_dmg_full = _dmg_full.dropna(subset=_dmg_required).copy()
_dmg_full = _dmg_full[_dmg_full['full_image_path'].apply(lambda p: p.exists())]
_dmg_full = _dmg_full.reset_index(drop=True)

print(f'Damage severity usable rows: {len(_dmg_full)}')
print(f'\nOld ({DMG_OLD_COL}):')
print(_dmg_full[DMG_OLD_COL].value_counts().to_string())
print(f'\nNew ({DMG_NEW_COL}):')
print(_dmg_full[DMG_NEW_COL].value_counts().to_string())
print(f'\nEvents:')
print(_dmg_full['disaster_type'].value_counts().to_string())

# ── Splits (no subsampling — use all ~2766 rows) ──────────────────────────────
_strat_d  = _dmg_full[DMG_OLD_COL] if can_stratify(_dmg_full[DMG_OLD_COL], 0.3) else None
_dmg_tr, _dmg_tmp = train_test_split(
    _dmg_full, test_size=0.3, random_state=SEED, stratify=_strat_d
)
_strat_d2 = _dmg_tmp[DMG_OLD_COL] if can_stratify(_dmg_tmp[DMG_OLD_COL], 0.5) else None
_dmg_vl, _dmg_te = train_test_split(
    _dmg_tmp, test_size=0.5, random_state=SEED, stratify=_strat_d2
)
_dmg_tr = _dmg_tr.reset_index(drop=True)
_dmg_vl = _dmg_vl.reset_index(drop=True)
_dmg_te = _dmg_te.reset_index(drop=True)
print(f'\nDamage splits → Train: {len(_dmg_tr)}, Val: {len(_dmg_vl)}, Test: {len(_dmg_te)}')

# ── BERT embeddings for damage task texts ─────────────────────────────────────
_all_dmg_texts = (
    pd.concat([_dmg_tr, _dmg_vl, _dmg_te])[TEXT_COLUMN]
    .fillna('').astype(str).tolist()
)
_dmg_missing = list({t for t in _all_dmg_texts if t not in bert_cache})
if _dmg_missing:
    print(f'\nEncoding {len(_dmg_missing)} new texts for damage task ...')
    _new_embs = _encode_batch(_dmg_missing)
    for _t, _e in zip(_dmg_missing, _new_embs):
        bert_cache[_t] = _e
    with open(BERT_CACHE, 'wb') as f:
        pickle.dump(bert_cache, f)
    print(f'BERT cache updated → {len(bert_cache)} total entries')
else:
    print('\nAll damage texts already in BERT cache.')

def _get_emb(df):
    return np.stack(
        [bert_cache[t] for t in df[TEXT_COLUMN].fillna('').astype(str)]
    ).astype(np.float32)

X_dmg_tr = _get_emb(_dmg_tr)
X_dmg_vl = _get_emb(_dmg_vl)
X_dmg_te = _get_emb(_dmg_te)

# ── Run damage severity benchmark ─────────────────────────────────────────────
DMG_LABEL_SETS = {
    'Old Annotations (image_damage)':     DMG_OLD_COL,
    'New Annotations (image_damage_llm)': DMG_NEW_COL,
}
TOTAL_DMG_RUNS       = len(CONFIGS) * len(DMG_LABEL_SETS)
dmg_run_counter      = 0
dmg_all_results      = []
dmg_all_event_results = []
dmg_bench_start      = time.time()

for label_name, label_col in DMG_LABEL_SETS.items():
    dmg_le = LabelEncoder()
    dmg_le.fit(
        pd.concat([_dmg_tr[label_col], _dmg_vl[label_col], _dmg_te[label_col]])
        .astype(str)
    )
    nc_dmg  = len(dmg_le.classes_)
    y_tr_d  = dmg_le.transform(_dmg_tr[label_col].astype(str))
    y_vl_d  = dmg_le.transform(_dmg_vl[label_col].astype(str))
    y_te_d  = dmg_le.transform(_dmg_te[label_col].astype(str))

    tr_ld, vl_ld, te_ld = make_loaders(
        y_tr_d, y_vl_d, y_te_d,
        tr_df=_dmg_tr, vl_df=_dmg_vl, te_df=_dmg_te,
        x_tr=X_dmg_tr, x_vl=X_dmg_vl, x_te=X_dmg_te,
    )

    print(f'\n{"#"*65}')
    print(f'  Annotation Set : {label_name}')
    print(f'  Classes ({nc_dmg})  : {dmg_le.classes_.tolist()}')
    print(f'{"#"*65}')

    for cfg_name, hp in CONFIGS.items():
        dmg_run_counter += 1
        run_start = time.time()
        print(f'\n  ┌─ [Run {dmg_run_counter}/{TOTAL_DMG_RUNS}] {cfg_name}')
        print(f'  │  d_model={hp["d_model"]}  lr={hp["lr"]}  '
              f'dropout={hp["dropout"]}  freeze={hp["freeze_image"]}  '
              f'epochs={EPOCHS}')
        print(f'  │')

        model = MultimodalFusion(
            text_dim=TEXT_DIM, num_classes=nc_dmg,
            d_model=hp['d_model'], n_heads=hp['n_heads'],
            dropout=hp['dropout'], freeze_image=hp['freeze_image'],
        ).to(DEVICE)

        dmg_metrics, y_true_t, y_pred_t = train_and_evaluate(
            model, tr_ld, vl_ld, te_ld, y_tr_d, nc_dmg, lr=hp['lr']
        )

        run_elapsed = time.time() - run_start
        dmg_all_results.append({
            'Annotation Set': label_name,
            'Model Config':   cfg_name,
            'D_MODEL':        hp['d_model'],
            'LR':             hp['lr'],
            'Dropout':        hp['dropout'],
            'Frozen':         hp['freeze_image'],
            **{k: round(v, 4) for k, v in dmg_metrics.items()},
        })

        # Per-event breakdown for damage task
        dmg_test_events = _dmg_te['disaster_type'].fillna('unknown').values
        for event in sorted(set(dmg_test_events)):
            mask = dmg_test_events == event
            n    = int(mask.sum())
            if n < 5:
                continue
            yt = [y_true_t[i] for i, m in enumerate(mask) if m]
            yp = [y_pred_t[i] for i, m in enumerate(mask) if m]
            dmg_all_event_results.append({
                'Event':          event,
                'Annotation Set': label_name,
                'Model Config':   cfg_name,
                'n_test':         n,
                'test_accuracy':  round(accuracy_score(yt, yp), 4),
                'test_wF1':       round(f1_score(yt, yp, average='weighted', zero_division=0), 4),
                'test_mF1':       round(f1_score(yt, yp, average='macro',    zero_division=0), 4),
            })

        print(f'  └─ Done in {run_elapsed:.0f}s  '
              f'[Acc={dmg_metrics["test_accuracy"]:.4f}  '
              f'wF1={dmg_metrics["test_wF1"]:.4f}  '
              f'mF1={dmg_metrics["test_mF1"]:.4f}]')
        del model

dmg_total_elapsed   = time.time() - dmg_bench_start
damage_results_df   = pd.DataFrame(dmg_all_results)
dmg_event_results_df = pd.DataFrame(dmg_all_event_results)

print(f'\n\n{"="*65}')
print(f'DAMAGE SEVERITY: ALL {TOTAL_DMG_RUNS} RUNS COMPLETE  '
      f'(total: {dmg_total_elapsed/60:.1f} min)')
print(f'{"="*65}')
print(damage_results_df[['Annotation Set', 'Model Config',
                          'test_accuracy', 'test_wF1', 'test_mF1']].to_string(index=False))
if not dmg_event_results_df.empty:
    print(f'\nPer-event records: {len(dmg_event_results_df)} '
          f'({dmg_event_results_df["Event"].nunique()} events)')


Damage severity usable rows: 2766

Old (image_damage):
image_damage
severe_damage              1525
mild_damage                 723
little_or_no_damage         433
dont_know_or_cant_judge      85

New (image_damage_llm):
image_damage_llm
severe_damage              2370
mild_damage                 194
little_or_no_damage         151
dont_know_or_cant_judge      51

Events:
disaster_type
hurricane_maria         902
hurricane_irma          813
hurricane_harvey        784
california_wildfires    250
iraq_iran_earthquake     17

Damage splits → Train: 1936, Val: 415, Test: 415

Encoding 2368 new texts for damage task ...
Loading distilbert-base-uncased ...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10159.39it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BERT encoding: 100%|██████████| 74/74 [00:19<00:00,  3.86it/s]


BERT cache updated → 2964 total entries

#################################################################
  Annotation Set : Old Annotations (image_damage)
  Classes (4)  : ['dont_know_or_cant_judge', 'little_or_no_damage', 'mild_damage', 'severe_damage']
#################################################################

  ┌─ [Run 1/8] A — Baseline
  │  d_model=256  lr=0.0001  dropout=0.2  freeze=True  epochs=15
  │
    Epoch 1/15 (73s) | train  loss=1.3507  acc=0.4375 | val    loss=1.3029  acc=0.4434  wF1=0.4392
    ↑ New best val wF1=0.4392 — checkpoint saved
    Epoch 2/15 (72s) | train  loss=1.2005  acc=0.5289 | val    loss=1.2357  acc=0.5687  wF1=0.5812
    ↑ New best val wF1=0.5812 — checkpoint saved
    Epoch 3/15 (71s) | train  loss=1.0794  acc=0.5522 | val    loss=1.2429  acc=0.5398  wF1=0.5429
    Epoch 4/15 (70s) | train  loss=1.0116  acc=0.5708 | val    loss=1.2665  acc=0.5783  wF1=0.5738
    Epoch 5/15 (70s) | train  loss=0.9460  acc=0.5930 | val    loss=1.2952  acc=0.522

In [ ]:
# ── Cell 10: Damage severity — comparison tables, charts, and standalone files
_DMG_OLD = 'Old Annotations (image_damage)'
_DMG_NEW = 'New Annotations (image_damage_llm)'
DMG_METRIC_LABELS = {'test_accuracy': 'Accuracy', 'test_wF1': 'Weighted F1', 'test_mF1': 'Macro F1'}

# ── 1. Overall comparison tables ─────────────────────────────────────────────
print('=' * 80)
print('DAMAGE SEVERITY RESULTS — Old vs New Annotations')
print('=' * 80)
for col, label in DMG_METRIC_LABELS.items():
    pivot = damage_results_df.pivot(index='Model Config', columns='Annotation Set', values=col)
    pivot.columns.name = None
    pivot.index.name   = None
    print(f'\n── {label} ──────────────────────────────────────────────────────')
    print(pivot.round(4).to_string())

# ── 2. Delta table ────────────────────────────────────────────────────────────
print('\n' + '=' * 80)
print('DELTA:  New – Old   (positive = new annotations helped)')
print('=' * 80)
dmg_delta_rows = []
for cfg in damage_results_df['Model Config'].unique():
    sub = damage_results_df[damage_results_df['Model Config'] == cfg].set_index('Annotation Set')
    if _DMG_OLD in sub.index and _DMG_NEW in sub.index:
        dmg_delta_rows.append({
            'Model Config':  cfg,
            'Δ Accuracy':    round(sub.loc[_DMG_NEW, 'test_accuracy'] - sub.loc[_DMG_OLD, 'test_accuracy'], 4),
            'Δ Weighted F1': round(sub.loc[_DMG_NEW, 'test_wF1']      - sub.loc[_DMG_OLD, 'test_wF1'],      4),
            'Δ Macro F1':    round(sub.loc[_DMG_NEW, 'test_mF1']      - sub.loc[_DMG_OLD, 'test_mF1'],      4),
        })
dmg_delta_df = pd.DataFrame(dmg_delta_rows)
print(dmg_delta_df.to_string(index=False))

# ── 3. Overall bar chart ──────────────────────────────────────────────────────
palette_dmg = {_DMG_OLD: '#4472C4', _DMG_NEW: '#ED7D31'}
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (metric_col, metric_label) in zip(axes, DMG_METRIC_LABELS.items()):
    pivot = damage_results_df.pivot(index='Model Config', columns='Annotation Set', values=metric_col)
    x     = np.arange(len(pivot))
    width = 0.35
    for j, (ann_set, color) in enumerate(palette_dmg.items()):
        if ann_set not in pivot.columns:
            continue
        bars = ax.bar(x + (j - 0.5) * width, pivot[ann_set], width,
                      label=ann_set, color=color, alpha=0.88, edgecolor='white')
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                    f'{h:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
    ax.set_title(metric_label, fontsize=12, fontweight='bold', pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels([c.replace(' — ', '\n') for c in pivot.index],
                       rotation=20, ha='right', fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(metric_label)
    ax.legend(fontsize=8, loc='lower right')
fig.suptitle(
    'Damage Severity Task: Old vs New Annotations — Model Performance Comparison',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
dmg_chart_path = RESULTS_DIR / 'damage_severity_comparison.png'
plt.savefig(dmg_chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\nOverall chart saved → {dmg_chart_path}')

# ── 4. Per-event damage severity tables + charts (if data available) ──────────
if not dmg_event_results_df.empty:
    _dmg_events = sorted(dmg_event_results_df['Event'].unique())
    dmg_ev_avg = (
        dmg_event_results_df
        .groupby(['Event', 'Annotation Set'])[list(DMG_METRIC_LABELS.keys())]
        .mean()
        .round(4)
        .reset_index()
    )

    print('\n' + '=' * 80)
    print('PER-EVENT RESULTS — Damage Severity Task  (averaged across 4 configs)')
    print('=' * 80)
    for metric_col, metric_label in DMG_METRIC_LABELS.items():
        pivot = dmg_ev_avg.pivot(index='Event', columns='Annotation Set', values=metric_col)
        pivot.columns.name = None
        pivot.index.name   = None
        if _DMG_OLD in pivot.columns and _DMG_NEW in pivot.columns:
            pivot['Δ (New−Old)'] = (pivot[_DMG_NEW] - pivot[_DMG_OLD]).round(4)
        print(f'\n── {metric_label} ──────────────────────────────────────────────')
        print(pivot.round(4).to_string())

    # Per-event × config delta table
    print('\n' + '=' * 80)
    print('PER-EVENT × CONFIG DELTA — Damage Accuracy  (New − Old)')
    print('=' * 80)
    dmg_ev_delta_rows = []
    for event in _dmg_events:
        row = {'Event': event}
        for cfg in CONFIGS.keys():
            sub = dmg_event_results_df[
                (dmg_event_results_df['Event'] == event) &
                (dmg_event_results_df['Model Config'] == cfg)
            ].set_index('Annotation Set')
            row[cfg] = round(
                sub.loc[_DMG_NEW, 'test_accuracy'] - sub.loc[_DMG_OLD, 'test_accuracy'], 4
            ) if (_DMG_OLD in sub.index and _DMG_NEW in sub.index) else float('nan')
        dmg_ev_delta_rows.append(row)
    dmg_ev_delta_df = pd.DataFrame(dmg_ev_delta_rows).set_index('Event')
    print(dmg_ev_delta_df.to_string())

    # Per-event bar chart
    fig3, axes3 = plt.subplots(1, 3, figsize=(20, 6))
    for ax, (metric_col, metric_label) in zip(axes3, DMG_METRIC_LABELS.items()):
        pivot = dmg_ev_avg.pivot(index='Event', columns='Annotation Set', values=metric_col)
        x     = np.arange(len(pivot))
        width = 0.35
        for j, (ann_set, color) in enumerate(palette_dmg.items()):
            if ann_set not in pivot.columns:
                continue
            bars = ax.bar(x + (j - 0.5) * width, pivot[ann_set], width,
                          label=ann_set, color=color, alpha=0.88, edgecolor='white')
            for bar in bars:
                h = bar.get_height()
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                        f'{h:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
        ax.set_title(metric_label, fontsize=12, fontweight='bold', pad=10)
        ax.set_xticks(x)
        ax.set_xticklabels([e.replace('_', '\n') for e in pivot.index],
                           rotation=20, ha='right', fontsize=8)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel(metric_label)
        ax.legend(fontsize=8, loc='lower right')
    fig3.suptitle(
        'Damage Severity Task: Per-Event Performance — Old vs New Annotations (Avg. across 4 Configs)',
        fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    dmg_ev_bar_path = RESULTS_DIR / 'per_event_damage_severity_comparison.png'
    plt.savefig(dmg_ev_bar_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nPer-event damage bar chart → {dmg_ev_bar_path}')

    # Per-event heatmap
    fig4, axes4 = plt.subplots(1, 3, figsize=(20, max(4, len(_dmg_events) * 0.9 + 1)))
    for ax, (metric_col, metric_label) in zip(axes4, DMG_METRIC_LABELS.items()):
        heat_data = {}
        for cfg in CONFIGS.keys():
            col_vals = []
            for event in _dmg_events:
                sub = dmg_event_results_df[
                    (dmg_event_results_df['Event'] == event) &
                    (dmg_event_results_df['Model Config'] == cfg)
                ].set_index('Annotation Set')
                col_vals.append(
                    round(sub.loc[_DMG_NEW, metric_col] - sub.loc[_DMG_OLD, metric_col], 4)
                    if (_DMG_OLD in sub.index and _DMG_NEW in sub.index) else float('nan')
                )
            heat_data[cfg.replace(' — ', '\n')] = col_vals
        heat_df = pd.DataFrame(heat_data, index=[e.replace('_', '\n') for e in _dmg_events])
        sns.heatmap(heat_df, ax=ax, annot=True, fmt='.3f', center=0,
                    cmap='RdYlGn', linewidths=0.5, linecolor='grey',
                    cbar_kws={'label': 'Δ (New−Old)'})
        ax.set_title(f'Δ {metric_label} (New − Old)', fontsize=11, fontweight='bold')
        ax.set_xlabel('')
    fig4.suptitle('Damage Severity Task: Per-Event Annotation Delta — New vs Old (Δ = New − Old)',
                  fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    dmg_ev_heat_path = RESULTS_DIR / 'per_event_damage_severity_delta_heatmap.png'
    plt.savefig(dmg_ev_heat_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Per-event damage heatmap  → {dmg_ev_heat_path}')

    # Save per-event CSVs
    dmg_ev_full_csv = RESULTS_DIR / 'per_event_damage_severity_results.csv'
    dmg_event_results_df.to_csv(dmg_ev_full_csv, index=False)
    print(f'Per-event damage CSV       → {dmg_ev_full_csv}')

    dmg_ev_avg_csv = RESULTS_DIR / 'per_event_damage_severity_avg.csv'
    dmg_ev_avg.to_csv(dmg_ev_avg_csv, index=False)
    print(f'Per-event avg damage CSV   → {dmg_ev_avg_csv}')

    dmg_ev_delta_csv = RESULTS_DIR / 'per_event_damage_severity_delta.csv'
    dmg_ev_delta_df.to_csv(dmg_ev_delta_csv)
    print(f'Per-event delta damage CSV → {dmg_ev_delta_csv}')
else:
    print('No per-event data for damage severity (all events < 5 test samples).')

# ── 5. Save overall damage CSVs ───────────────────────────────────────────────
dmg_csv_path = RESULTS_DIR / 'damage_severity_results.csv'
damage_results_df.to_csv(dmg_csv_path, index=False)
print(f'\nOverall damage CSV → {dmg_csv_path}')

dmg_delta_csv_path = RESULTS_DIR / 'damage_severity_delta.csv'
dmg_delta_df.to_csv(dmg_delta_csv_path, index=False)
print(f'Damage delta CSV   → {dmg_delta_csv_path}')


DAMAGE SEVERITY RESULTS — Old vs New Annotations

── Accuracy ──────────────────────────────────────────────────────
                       New Annotations (image_damage_llm)  Old Annotations (image_damage)
A — Baseline                                       0.8000                          0.5325
B — Larger Model                                   0.6964                          0.5253
C — Unfrozen Backbone                              0.8313                          0.5566
D — Larger + Unfrozen                              0.8434                          0.5446

── Weighted F1 ──────────────────────────────────────────────────────
                       New Annotations (image_damage_llm)  Old Annotations (image_damage)
A — Baseline                                       0.8085                          0.5477
B — Larger Model                                   0.7441                          0.5280
C — Unfrozen Backbone                              0.8269                          0.5614
D 